# 2.4 — Feature Scaling

Feature scaling brings all numeric columns to the same range so no single column dominates just because its numbers are bigger.

> **Why it matters:** Algorithms that use distance (KNN, SVM) or gradients (Linear Regression, Neural Nets) are sensitive to scale. A salary column (0–90,000) will completely overshadow an age column (0–30) without scaling.

### Two scalers:
- **StandardScaler** — mean=0, std=1. Default choice. Handles outliers well.
- **MinMaxScaler** — squeezes values to 0–1. Use for neural networks or when you need a strict 0–1 range.

### Golden rule:
> **Split first. Scale after. Fit only on train.**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

data = {
    'age':    [22, 25, 19, 23, 26, 21, 28, 24],
    'salary': [45000, 62000, 38000, 91000, 54000, 42000, 76000, 49000],
    'score':  [88, 72, 45, 91, 66, 78, 85, 70],
    'passed': [1, 1, 0, 1, 1, 1, 1, 0]
}

df = pd.DataFrame(data)
df

## 1. The Problem — Unscaled Features

See how different the ranges are across columns:

In [ ]:
X = df.drop(columns=['passed'])
y = df['passed']

print("Ranges before scaling:")
print(X.describe().loc[['min','max']])

## 2. Split First — Always

Split before any scaling or processing. The test set must never influence the scaler.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)

## 3. StandardScaler

Transforms each column to mean=0 and std=1.

Formula: `z = (value - mean) / std` — same as the z-score from 2.2, different purpose.

- `fit()` — memorises mean and std from training data only
- `transform()` — applies the formula using those memorised values
- `fit_transform()` — does both in one step (only use on train)

In [ ]:
scaler_std = StandardScaler()

# fit_transform on train — learns stats AND scales
X_train_std = scaler_std.fit_transform(X_train)

# transform only on test — uses train's stats
X_test_std = scaler_std.transform(X_test)

print("Mean memorised by scaler:", scaler_std.mean_.round(2))
print("Std memorised by scaler: ", scaler_std.scale_.round(2))

In [ ]:
# Convert back to DataFrame for readability
X_train_std_df = pd.DataFrame(X_train_std, columns=X.columns)
print("After StandardScaler:")
print(X_train_std_df.round(2))

print("\nMean of each column (should be ~0):")
print(X_train_std_df.mean().round(4))

print("\nStd of each column (should be ~1):")
print(X_train_std_df.std().round(4))

## 4. MinMaxScaler

Squeezes all values to exactly 0–1.

Formula: `x_scaled = (x - x_min) / (x_max - x_min)`

Warning: sensitive to outliers — one extreme value compresses everything else.

In [ ]:
scaler_mm = MinMaxScaler()

X_train_mm = scaler_mm.fit_transform(X_train)
X_test_mm  = scaler_mm.transform(X_test)

X_train_mm_df = pd.DataFrame(X_train_mm, columns=X.columns)
print("After MinMaxScaler:")
print(X_train_mm_df.round(2))

print("\nMin of each column (should be 0):")
print(X_train_mm_df.min())

print("\nMax of each column (should be 1):")
print(X_train_mm_df.max())

## 5. Data Leakage — What NOT to Do

Data leakage = the model accidentally sees test/future information during training. It looks great during testing but fails in production.

In [ ]:
# WRONG — fitting on full dataset before splitting
# The scaler sees test data's mean/std — leakage!
scaler_bad = StandardScaler()
X_scaled_bad = scaler_bad.fit_transform(X)       # leakage here
X_train_bad, X_test_bad = train_test_split(X_scaled_bad)  # too late

print("WRONG: scaler mean from full data:", scaler_bad.mean_.round(2))
print("CORRECT: scaler mean from train only:", scaler_std.mean_.round(2))
print("\nThey differ because the wrong scaler saw test data too.")

## 6. Which Algorithms Need Scaling?

| Algorithm | Needs scaling? | Why |
|-----------|---------------|-----|
| KNN | Yes — critical | Uses distance between points |
| SVM | Yes — critical | Uses distance to hyperplane |
| Logistic Regression | Yes | Uses gradient descent |
| Linear Regression | Yes | Uses gradient descent |
| Neural Networks | Yes | Gradient-based |
| PCA | Yes | Variance-based |
| Decision Tree | No | Splits on thresholds, not distances |
| Random Forest | No | Ensemble of trees |
| Naive Bayes | No | Probability-based |

> **Default:** Use StandardScaler for everything except tree-based models.

## 7. Decision Rules

| Situation | Scaler |
|-----------|--------|
| General purpose, default choice | StandardScaler |
| Need strict 0–1 range, no outliers | MinMaxScaler |
| Neural networks | MinMaxScaler |
| Tree models (RF, Decision Tree) | No scaling needed |

> **Key insight:** Same z-score formula from 2.2 — different purpose. In 2.2 you used it to detect outliers. Here you use it to bring all columns to the same scale.

## Practice Task

In [ ]:
practice_data = {
    'age':    [22, 25, 19, 23, 26, 21, 28, 24, 30, 27],
    'salary': [45000, 62000, 38000, 91000, 54000, 42000, 76000, 49000, 83000, 57000],
    'score':  [88, 72, 45, 91, 66, 78, 85, 70, 92, 74],
    'passed': [1, 1, 0, 1, 1, 1, 1, 0, 1, 1]
}
practice_df = pd.DataFrame(practice_data)
X_p = practice_df.drop(columns=['passed'])
y_p = practice_df['passed']

# Q1 — Split X_p and y_p into train/test (test_size=0.2, random_state=42)
# YOUR CODE HERE

# Q2 — Apply StandardScaler. Fit on train, transform both.
# YOUR CODE HERE

# Q3 — Apply MinMaxScaler on the same split. Compare output ranges.
# YOUR CODE HERE

# Q4 — Print mean and std of each column after StandardScaler on train.
# Should be ~0 and ~1 respectively.
# YOUR CODE HERE